In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('data/raw/telco_churn.csv')
print(f"Original shape: {df.shape}")
print(f"\nColumn dtypes:\n{df.dtypes}")

Original shape: (7043, 21)

Column dtypes:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object


In [3]:
# find and handle nulls

# Check nulls before fix
print("Nulls BEFORE fix:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Fix TotalCharges — convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check which rows became null
null_rows = df[df['TotalCharges'].isnull()]
print(f"\nRows with null TotalCharges: {len(null_rows)}")
print(null_rows[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']])

Nulls BEFORE fix:
Series([], dtype: int64)

Rows with null TotalCharges: 11
      customerID  tenure  MonthlyCharges  TotalCharges
488   4472-LVYGI       0           52.55           NaN
753   3115-CZMZD       0           20.25           NaN
936   5709-LVOEQ       0           80.85           NaN
1082  4367-NUYAO       0           25.75           NaN
1340  1371-DWPAZ       0           56.05           NaN
3331  7644-OMVMY       0           19.85           NaN
3826  3213-VVOLG       0           25.35           NaN
4380  2520-SGTTA       0           20.00           NaN
5218  2923-ARZLG       0           19.70           NaN
6670  4075-WKNIU       0           73.35           NaN
6754  2775-SEFEE       0           61.90           NaN


In [4]:
#  — fix the nulls
# These 11 customers have tenure=0 so TotalCharges should be 0
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Verify
print("Nulls AFTER fix:")
print(df.isnull().sum().sum(), "total nulls remaining")
print("\nTotalCharges dtype:", df['TotalCharges'].dtype)

Nulls AFTER fix:
0 total nulls remaining

TotalCharges dtype: float64


In [5]:
# = fix the nulls
# These 11 customers have tenure=0 so TotalCharges should be 0
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Verify
print("Nulls AFTER fix:")
print(df.isnull().sum().sum(), "total nulls remaining")
print("\nTotalCharges dtype:", df['TotalCharges'].dtype)

Nulls AFTER fix:
0 total nulls remaining

TotalCharges dtype: float64


In [6]:
#  SeniorCitizen is stored as 0/1 integer but needs label for clarity
print("SeniorCitizen before:", df['SeniorCitizen'].unique())

df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

print("SeniorCitizen after:", df['SeniorCitizen'].unique())

SeniorCitizen before: [0 1]
SeniorCitizen after: <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str


In [7]:
# convert all Yes/No columns to 1/0
binary_cols = ['Partner', 'Dependents', 'PhoneService',
               'PaperlessBilling', 'SeniorCitizen']

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})
    print(f"{col}: {df[col].unique()}")

# Encode target column
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print(f"\nChurn: {df['Churn'].unique()}")
print(f"Churn rate: {df['Churn'].mean()*100:.1f}%")

Partner: [1 0]
Dependents: [0 1]
PhoneService: [0 1]
PaperlessBilling: [1 0]
SeniorCitizen: [0 1]

Churn: [0 1]
Churn rate: 26.5%


In [8]:
#  some columns have 3 values including 'No internet service'
# These mean the same as 'No' — simplify them

three_val_cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                  'DeviceProtection', 'TechSupport',
                  'StreamingTV', 'StreamingMovies']

for col in three_val_cols:
    print(f"{col}: {df[col].unique()}")
    # Replace 'No internet service' and 'No phone service' with 'No'
    df[col] = df[col].replace({'No internet service': 'No',
                                'No phone service': 'No'})
    # Now encode Yes/No → 1/0
    df[col] = df[col].map({'Yes': 1, 'No': 0})
    print(f"  → After: {df[col].unique()}\n")

MultipleLines: <ArrowStringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str
  → After: [0 1]

OnlineSecurity: <ArrowStringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
  → After: [0 1]

OnlineBackup: <ArrowStringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str


  → After: [1 0]

DeviceProtection: <ArrowStringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
  → After: [0 1]

TechSupport: <ArrowStringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
  → After: [0 1]

StreamingTV: <ArrowStringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
  → After: [0 1]

StreamingMovies: <ArrowStringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
  → After: [0 1]



In [9]:
# Drop customerID — unique identifier, no predictive value
df = df.drop(columns=['customerID'])
print(f"Shape after dropping customerID: {df.shape}")

# Encode gender — Male=1, Female=0
df['gender'] = (df['gender'] == 'Male').astype(int)
print(f"gender encoded: {df['gender'].unique()}")

Shape after dropping customerID: (7043, 20)
gender encoded: [0 1]


In [10]:
print("="*45)
print("  CLEAN DATASET SUMMARY")
print("="*45)
print(f"  Shape         : {df.shape}")
print(f"  Nulls         : {df.isnull().sum().sum()}")
print(f"  Churn rate    : {df['Churn'].mean()*100:.1f}%")
print()
print("  Numeric cols:")
num_cols = df.select_dtypes(include=np.number).columns.tolist()
for c in num_cols:
    print(f"    {c}")
print()
print("  Remaining text cols (for OneHotEncoder in Week 2):")
obj_cols = df.select_dtypes(include='object').columns.tolist()
for c in obj_cols:
    print(f"    {c}: {df[c].unique().tolist()}")
print("="*45)

  CLEAN DATASET SUMMARY
  Shape         : (7043, 20)
  Nulls         : 0
  Churn rate    : 26.5%

  Numeric cols:
    gender
    SeniorCitizen
    Partner
    Dependents
    tenure
    PhoneService
    MultipleLines
    OnlineSecurity
    OnlineBackup
    DeviceProtection
    TechSupport
    StreamingTV
    StreamingMovies
    PaperlessBilling
    MonthlyCharges
    TotalCharges
    Churn

  Remaining text cols (for OneHotEncoder in Week 2):
    InternetService: ['DSL', 'Fiber optic', 'No']
    Contract: ['Month-to-month', 'One year', 'Two year']
    PaymentMethod: ['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)']


C:\Users\HP\AppData\Local\Temp\ipykernel_4596\2681593083.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(include='object').columns.tolist()


In [13]:
df.to_csv('data/processed/telco_churn_clean.csv', index=False)
print("Saved: data/processed/telco_churn_clean.csv")

# Quick verification
df_verify = pd.read_csv('data/processed/telco_churn_clean.csv')
print(f"Verified shape: {df_verify.shape}")


Saved: data/processed/telco_churn_clean.csv
Verified shape: (7043, 20)


In [14]:
# Day 5 — Data Cleaning Notes
notes = """
# Day 5 — Data Cleaning Notes

## What was done today

### 1. Fixed TotalCharges dtype
- Problem : Stored as `object` (text) instead of `float64`
- Fix     : pd.to_numeric(df['TotalCharges'], errors='coerce')
- Result  : 11 rows became NaN (all had tenure=0)

### 2. Handled 11 Null Rows
- Problem : 11 customers had blank TotalCharges
- Reason  : tenure=0 means brand new customer, never billed
- Fix     : Filled with 0 (TotalCharges = 0 for new customers)
- Result  : 0 nulls remaining in entire dataset

### 3. Dropped customerID
- Reason  : Unique identifier per row, no predictive value
- Result  : Columns reduced from 21 → 20

### 4. Fixed SeniorCitizen
- Problem : Stored as 0/1 integer, inconsistent with other columns
- Fix     : Mapped 0→No, 1→Yes, then re-encoded Yes→1, No→0
- Result  : Consistent encoding across all binary columns

### 5. Encoded Binary Yes/No Columns → 0/1
Columns encoded:
- Partner, Dependents, PhoneService, PaperlessBilling, SeniorCitizen
- gender (Male=1, Female=0)
- Churn target (Yes=1, No=0)

### 6. Simplified Three-Value Columns → 0/1
Columns affected:
- MultipleLines, OnlineSecurity, OnlineBackup
- DeviceProtection, TechSupport, StreamingTV, StreamingMovies

Fix: 'No internet service' and 'No phone service' both mapped to 'No'
Then encoded Yes→1, No→0

### 7. Left Multi-Class Columns as Text
Columns kept as text for OneHotEncoder in Week 2:
- InternetService  : DSL / Fiber optic / No
- Contract         : Month-to-month / One year / Two year
- PaymentMethod    : Electronic check / Mailed check / etc.

## Final Clean Dataset Summary
- Shape      : (7043, 20)
- Nulls      : 0
- Churn rate : 26.5%
- Saved to   : data/processed/telco_churn_clean.csv

## Key Decisions Made
| Decision | Reason |
|----------|--------|
| Filled nulls with 0 not dropped | tenure=0 customers are valid data points |
| Kept 3 text columns as-is | OneHotEncoder handles them better in pipeline |
| Dropped customerID | No signal, just a row identifier |

## Tomorrow — Day 6 (Feature Engineering)
- Create numServices  : count of all services subscribed per customer
- Create chargePerMonth: TotalCharges / (tenure + 1)
- Create tenureBucket : group tenure into 4 buckets (New/Growing/Mature/Loyal)
- Save final feature-engineered dataset
"""

with open('reports/eda_notes.md', 'a' ,encoding="utf-8" ) as f:
    f.write(notes)

print("Day 5 notes saved to reports/day5_cleaning_notes.md")

Day 5 notes saved to reports/day5_cleaning_notes.md
